- `Acesso aos dados`: [MCD19A2.061: Terra & Aqua MAIAC Land Aerosol Optical Depth Daily 1km ](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD19A2_GRANULES)
- `Período dos dados`: 2000-02-24T00:00:00Z–2024-05-16T23:55:00Z
- `Resolução espacial`: 1000 metros
- `Resolução temporal`: diária
- `Variável utilizada`: Aerosol optical depth over land retrieved in the MODIS Green band (0.55 μm)
- `Código realizado por`: Enrique V. Mattos - 13/04/2025

# **1° Passo:** Preparando ambiente

In [ ]:
# instalando bibliotecas
!pip install -q ultraplot cartopy salem rasterio

# iniciando GEE e instalando XEE (transforma dados do GEE para formato DataSet)
!pip install -q eemont xee
import ee, geemap
ee.Authenticate()
ee.Initialize(project='ee-enrique', opt_url='https://earthengine-highvolume.googleapis.com')

# importa bibliotecas
import numpy as np
import ultraplot as uplt
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
from zipfile import ZipFile
import salem
from datetime import datetime, timedelta
import glob
import xarray as xr
import os
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings('ignore')

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# caminho do drive
dir = '/content/drive/MyDrive/5_EXTENSAO/02_prefeitura_analise_periodo_chuvoso_2024_2025'

# leitura do shapefile do Brasil
shapefile_brasil = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp')

# leitura do shapefile com a biblioteca SALEM
url = 'https://github.com/evmpython/shapefile/raw/main/'
shp = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
itajuba = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
mg = salem.read_shapefile(f'{url}estado_MG/MG_UF_2019.shp')

# limites do Brasil
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

# limites de MG
lonmin_MG, lonmax_MG, latmin_MG, latmax_MG = -52., -39., -23., -14.

# **PARTE 1):** Processamento

## 1) Mapa no GEE

In [ ]:
#========================================================================================================================#
#                                          FILTRA REGIÃO DE INTERESSE
#========================================================================================================================#
brasil = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
municipio_itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

#========================================================================================================================#
#                                            CARREGA OS DADOS
#========================================================================================================================#
# carrega os dados
AOD_055 = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
            .filter(ee.Filter.date('2024-10-01', '2024-11-01')) \
            .select('Optical_Depth_055') \
            .filterBounds(municipio_itajuba)

#========================================================================================================================#
#                                           PLOTA FIGURA
#========================================================================================================================#
# cria a moldura do mapa
Map = geemap.Map()

# centraliza o mapa na região
Map.centerObject(municipio_itajuba, zoom=11)

# parâmetros de visualização
vis = {'min': 0, 'max': 1, 'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']}

# plota mapa
Map.addLayer(AOD_055.max().clip(municipio_itajuba).multiply(0.001), vis, 'AOD MODIS Green band (0.55 μm)')

# contorno da região
style1 = {'color': 'red', 'fillColor': '00000000'}
Map.addLayer(municipio_itajuba.style(**style1), {}, 'MG')

# barra de cores
Map.add_colorbar_branca(colors=vis['palette'], vmin=vis['min'], vmax=vis['max'], layer_name='AOD MODIS Green band (0.55 μm)')

# exibe na tela
Map

In [ ]:
# mostra os dados
AOD_055

Em 1 mês de dados tem 2575 arquivos

In [ ]:
# transforma a data para o formato "ano-mes-dia hora:minuto" e extrai os valores
AOD_055_2 = AOD_055.map(lambda img: img.set( {"DATE": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd hh:mm")}))

# agrega as informações
agrega = (AOD_055_2.aggregate_array("DATE").getInfo())

# mostra na tela
pd.DataFrame(agrega)

## 2) Produz arquivo netcdf
- Demorou `2min 47s` para gerar 6 arquivos netcdf
- Em 6 meses de dados de AOD para MG existem 17593 arquivos

In [ ]:
%%time
#========================================================================================================================#
#                                          FILTRA REGIÃO DE INTERESSE
#========================================================================================================================#
brasil = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
municipio_itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

#========================================================================================================================#
#                                           DEFINE A DATA INICIAL E FINAL
#========================================================================================================================#
data_inicial = '2024-10-01'
data_final = '2025-04-01'

#========================================================================================================================#
#                                               CARREGA OS DADOS
#========================================================================================================================#
# carrega os dados. Período entre "2023-10-01" à "2024-04-01", ou seja do mês 10/2023 à 03/2024
AOD_055 = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
            .filter(ee.Filter.date(data_inicial, data_final)) \
            .select('Optical_Depth_055') \
            .filterBounds(estado_mg)

#========================================================================================================================#
#                                          PRODUZ ARQUIVO NETCDF
#========================================================================================================================#
# Loop entre os meses
for file in pd.date_range(data_inicial.replace('-', ''), data_final.replace('-', ''), freq='1M'):

    print('#-------------------------------------------#')
    print('.... PROCESSANDO', file)
    print('#-------------------------------------------#')

    # extrai ano e mês
    ano = file.strftime('%Y')
    mes = file.strftime('%m')

    # monta data no formato "2024-03-31"
    date = f"{ano}-{mes}-{file.strftime('%d')}"

    # data em formato do GEE
    data_ee = ee.Date(date)

    # extrai o range por intervalo de mês. Exemplo: "DateRange [2024-03-01 00:00:00, 2024-04-01 00:00:00]""
    range = data_ee.getRange('month')

    # calcula a média mensal
    AOD_055_mes = AOD_055.filter(ee.Filter.date(range)).max().clip(estado_mg).multiply(0.001).set({'system:time_start': data_ee.millis(), 'mes': data_ee.format('YYYY-MM')})

    # converte para Dataset
    ds_AOD_055_mensal = xr.open_dataset(AOD_055_mes,
                                        engine = 'ee',
                                        crs = 'EPSG:4326',
                                        scale = 0.10,
                                        geometry = estado_mg.geometry())

    # muda de "(time, lon, lat)" para "(time, lat, lon)"
    ds_AOD_055_mensal = ds_AOD_055_mensal.transpose("time", "lat", "lon")

    # salva para arquivo netcdf
    ds_AOD_055_mensal.to_netcdf(f'{dir}/output/03_AOD/AOD_055_mensal_{ano}-{mes}.nc')

In [ ]:
AOD_055

In [ ]:
# transforma a data para o formato "ano-mes-dia hora:minuto" e extrai os valores
AOD_055_2 = AOD_055.map(lambda img: img.set( {"DATE": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd hh:mm")}))

# agrega as informações
agrega = (AOD_055_2.aggregate_array("DATE").getInfo())

# mostra na tela
pd.DataFrame(agrega)

In [ ]:
# exemplo do arquivo netcdf gerado
ds_AOD_055_mensal

In [ ]:
ds_AOD_055_mensal['Optical_Depth_055'].plot(vmax=0.5)

# **PARTE 2):** Plota painel de figuras - `MAPA DE DENSIDADE`

In [ ]:
# lista dos arquivos mensais. Exemplo: "S5P_co_mensal_2023_11.nc"
files = sorted(glob.glob(f'{dir}/output/03_AOD/AOD_055_mensal_*nc'))

# meses
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

# cria a moldura da figura
fig, ax = uplt.subplots(figsize=(18, 12.5), nrows=2, ncols=3, tight=True, proj='pcarree', sharex=True, sharey=True)

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=False, latlines=5, lonlines=10,
          latlim=(latmin_MG, latmax_MG), lonlim=(lonmin_MG, lonmax_MG),
          suptitle=f'Aerosol optical depth over land retrieved in the MODIS Green band (0.55 μm)',
          small='20px', large='35px',
          linewidth=0, grid=False)

# loop dos anos
for i, file in enumerate(files):

    # leitura do arquivo netcdf
    ds = xr.open_dataset(file)

    # nome do arquivo
    basename = os.path.basename(os.path.splitext(file)[0])

    # extrai o ano e mês da imagem
    ano, mes = basename[15:19], basename[20:22]

    print('Processando ===>>>',  ano, mes)

    # plota figura
    # opções de palletes: https://proplot.readthedocs.io/en/latest/colormaps.html
    map1 = ax[i].contourf(ds['lon'],
                          ds['lat'],
                          ds['Optical_Depth_055'][0,:,:].salem.roi(shape=mg),
                          cmap='lajolla',
                          vmin=0.0, vmax=0.4,
                          levels=uplt.arange(0.0, 0.4, 0.1),
                          extend='max')

    # total de focos no estado de MG
    total = np.max(ds['Optical_Depth_055'][0,:,:].salem.roi(shape=mg))
    total = round(float(total.values), 1)
    nome_subtitulo = f'{meses[int(mes)-1]}/{ano}={total}'

    # plota titulo de cada figura
    ax[i].format(title=nome_subtitulo, labels = False, titleloc='c', titlecolor='bright blue', fontsize=20)

    # plota contorno de MG e Itajubá
    mg.plot(edgecolor='black', facecolor='none', linewidth=1.2, alpha=1, ax=ax[i])
    itajuba.plot(edgecolor='red', facecolor='none', linewidth=0.5, alpha=1, ax=ax[i])

# informação na figura
ax[3].annotate('Prof. Enrique Mattos/UNIFEI\ngithub.com/evmpython', xy=(lonmin_MG, latmin_MG-4.0), fontsize=15, color='black')

# plota barra de cores da figura
fig.colorbar(map1, loc='b', label='Fonte: AQUA e TERRA/Pixel: 10km', ticks=0.1, ticklabelsize=22, labelsize=22, space=0.5, length=0.60, width=0.4)

# salva figura
fig.savefig(f'{dir}/output/Fig_6_AOD_055_MG.jpg', transparent=True, dpi=300, bbox_inches="tight", pad_inches=0.1)

In [ ]:
ds